# 02 — Build and Run an Agent

This notebook walks you through creating an agent, running conversations,
and observing tool calls — all locally using the Microsoft Agent Framework.
Run cells top-to-bottom. Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [ ]:
import sys, pathlib

# Ensure the local project root is first on sys.path so our `agents`
# package takes priority over the openai-agents site-package.
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Change this to work with a different agent
AGENT_NAME = "code-helper"

## Install Dependencies

In [ ]:
!uv sync

## Connect and Create Agent

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Reset cached credential so it picks up AZURE_AUTHORITY_HOST from .env
from agents._base.client import reset_credential
reset_credential()

from agents.registry import REGISTRY
from agents._base.agent_factory import create_agent

entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()
agent = create_agent(config)

print(f"✓ Agent '{config.agent_name}' assembled")
print(f"  Model:   {config.agent_deployment_name}")
print(f"  Endpoint: {config.azure_ai_project_endpoint[:50]}…")


## Step 1: Inspect Agent Tools

See what tools the agent has available. These are plain Python functions
that the Agent Framework auto-wraps and calls during conversations.

In [ ]:
import importlib

module_name = AGENT_NAME.replace("-", "_")
try:
    tools_module = importlib.import_module(f"agents.{module_name}.tools")
    tools = getattr(tools_module, "TOOLS", [])
    if tools:
        print(f"Agent '{AGENT_NAME}' has {len(tools)} tool(s):")
        for tool in tools:
            name = tool.__name__ if hasattr(tool, "__name__") else str(tool)
            doc = (tool.__doc__ or "No description").strip().split("\n")[0]
            print(f"  • {name}: {doc}")
    else:
        print(f"Agent '{AGENT_NAME}' is instruction-only (no tools)")
except ModuleNotFoundError:
    print(f"No tools module found for '{AGENT_NAME}'")

## Step 2: Send a Message

Run a single-turn conversation. The Agent Framework handles tool calling
automatically — you just send a prompt and get back a response.

In [15]:
# Change this to test different prompts
USER_MESSAGE = "Hello! What can you help mw with?"

result = await agent.run(USER_MESSAGE)
response_text = result.text if hasattr(result, "text") else str(result)

print(f"User: {USER_MESSAGE}")
print(f"\nAgent: {response_text}")

User: Hello! What can you help mw with?

Agent: Hi, mw! I can help you with coding tasks like writing or debugging code, explaining programming concepts, suggesting best practices, or assisting with design patterns. Let me know what you're working on!


## Step 3: Trigger a Tool Call

Send a prompt that should trigger the agent's greeting tool.
The Agent Framework calls the tool function automatically and
incorporates the result into the response.

In [ ]:
# This should trigger the greet_user tool (for code-helper)
TOOL_MESSAGE = "Please greet Alice using your greeting tool."

result = await agent.run(TOOL_MESSAGE)
response_text = result.text if hasattr(result, "text") else str(result)

print(f"User: {TOOL_MESSAGE}")
print(f"\nAgent: {response_text}")

## Step 4: Multi-Turn Conversation

Use a loop for an interactive multi-turn conversation.
Type `quit` or `exit` to stop.

In [ ]:
print(f"Chat with '{AGENT_NAME}' — type 'quit' to stop\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", ""):
        print("\n✓ Conversation ended")
        break

    # Create a fresh agent for each turn (stateless)
    agent = create_agent(config)
    result = await agent.run(user_input)
    response_text = result.text if hasattr(result, "text") else str(result)
    print(f"Agent: {response_text}\n")

## Step 5: Use the Programmatic API

The `agents` package provides a public API for scripting. The `run_agent()` function
is a sync convenience wrapper that creates the agent, runs the prompt, and returns
the response text.

In [ ]:
from agents import run_agent

response = run_agent(config, "Explain what a Python decorator is in one sentence.")
print(f"Response: {response}")

## Step 6: Try a Different Agent

Switch to `doc-assistant` (or any other registered agent) and run a prompt.

In [14]:
other_agent = "doc-assistant"
other_entry = REGISTRY.get_agent(other_agent)
other_config = other_entry.config_class()
other = create_agent(other_config)

result = await other.run("What makes a good README?")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"'{other_agent}' says:\n{response_text}")

SSL verification disabled (DISABLE_SSL_VERIFY=true). Do NOT use this in production.


'doc-assistant' says:
A good README is comprehensive, clear, and organized. It should provide all the necessary information for users to understand and use the project while maintaining simplicity. Here are the key elements to include in an effective README:

### **1. Project Title and Description**
- **Title:** The name of your project, presented prominently.
- **Description:** A brief summary of the project purpose, features, and why it exists. Aim to make it engaging and informative.

### **2. Table of Contents**
- Include a table of contents for easy navigation if the README is lengthy.

### **3. Installation Instructions**
- Detailed steps to install and set up the project. Specify dependencies and provide clear commands for installation.
  - Example:
    ```
    git clone https://github.com/user/project-name.git
    cd project-name
    npm install
    ```

### **4. Usage**
- Show users how to use your project with examples, commands, or screenshots.
  - Example:
    ```
    To st

## Next Steps

- Change the prompt in Step 2 or Step 3 to test different interactions
- Try a different agent: change `AGENT_NAME` at the top and re-run all cells
- Add custom tools — see the [Custom Tools Guide](../docs/custom-tools-guide.md)
- See **03_scaffold_agent.ipynb** to create a brand new agent from scratch